# RoadSafe India — Population Density Data

## Objective

Evaluate State/UT population density as a potential explanatory variable
for road accident and fatality burden.

## Primary Variable

Population density (persons per square kilometre)

## Research Questions

1. Is population density associated with accidents per 100,000 population?
2. Is population density associated with fatalities per 100,000 population?
3. Does population density contribute additional explanatory information
   beyond vehicle exposure?

## Data Requirements

The selected source must provide:

- State/UT-level values
- Consistent geographic definitions
- Clearly documented reference year
- Reliable official or authoritative provenance

## Important Limitation

Population density is a contextual demographic indicator and does not
directly measure traffic exposure, congestion or road usage.

In [4]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent

density_path = (
    project_root
    / "data"
    / "research"
    / "census_2011_population_area.xlsx"
)

print("File exists:", density_path.exists())

excel = pd.ExcelFile(density_path)

print("\nAvailable sheets:")
for sheet in excel.sheet_names:
    print(sheet)

File exists: True

Available sheets:
Sheet1


In [5]:
# -------------------------------------------------------
# Inspect Census 2011 workbook
# -------------------------------------------------------

density_raw = pd.read_excel(
    density_path,
    sheet_name="Sheet1",
    header=None
)

print("Shape:", density_raw.shape)

print("\nFirst 20 rows:")

display(
    density_raw.head(20)
)

Shape: (20024, 15)

First 20 rows:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,"A-1 NUMBER OF VILLAGES, TOWNS, HOUSEHOLDS, POP...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,State Code,District Code,Sub District Code,India/ State/ Union Territory/ District/ Sub-d...,Name,Total/\nRural/\nUrban,Number of villages,NaN,Number of towns,Number of households,Population,NaN,NaN,Area\n (In sq. km),Population per sq. km.
2,NaN,NaN,NaN,NaN,NaN,NaN,Inhabited,Uninhabited,NaN,NaN,Persons,Males,Females,NaN,NaN
3,1,2,3,4,5,6,7,8,9,10,11,12,13,13,14
4,00,000,00000,INDIA,INDIA @&,Total,597608,43324,7933,249501663,1210854977,623270258,587584719,3287469,382
5,00,000,00000,INDIA,INDIA $,Rural,597608,43324,0,168612897,833748852,427781058,405967794,3101473.97,279
6,00,000,00000,INDIA,INDIA $,Urban,0,0,7933,80888766,377106125,195489200,181616925,102252.03,3685
7,01,000,00000,STATE,JAMMU & KASHMIR @&,Total,6337,216,122,2119718,12541302,6640662,5900640,222236,124
8,01,000,00000,STATE,JAMMU & KASHMIR,Rural,6337,216,0,1553433,9108060,4774477,4333583,220990.1,91
9,01,000,00000,STATE,JAMMU & KASHMIR,Urban,0,0,122,566285,3433242,1866185,1567057,1245.9,2755


In [6]:
# -------------------------------------------------------
# Print rows with their row numbers
# -------------------------------------------------------

for i in range(
    min(20, len(density_raw))
):
    print(
        i,
        "→",
        density_raw.iloc[i].tolist()
    )

0 → ['A-1 NUMBER OF VILLAGES, TOWNS, HOUSEHOLDS, POPULATION AND AREA', nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
1 → ['State  Code', 'District Code', 'Sub District Code', 'India/ State/ Union Territory/ District/ Sub-district', 'Name', 'Total/\nRural/\nUrban', 'Number of villages', nan, 'Number of towns', 'Number of households', 'Population', nan, nan, 'Area\n (In sq. km)', 'Population per sq. km.']
2 → [nan, nan, nan, nan, nan, nan, 'Inhabited', 'Uninhabited', nan, nan, 'Persons', 'Males', 'Females', nan, nan]
3 → [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 13, 14]
4 → ['00', '000', '00000', 'INDIA', 'INDIA @&', 'Total', 597608, 43324, 7933, 249501663, 1210854977, 623270258, 587584719, 3287469, 382]
5 → ['00', '000', '00000', 'INDIA', 'INDIA $', 'Rural', 597608, 43324, 0, 168612897, 833748852, 427781058, 405967794, 3101473.97, 279]
6 → ['00', '000', '00000', 'INDIA', 'INDIA $', 'Urban', 0, 0, 7933, 80888766, 377106125, 195489200, 181616925, 102252.03, 3685]

In [7]:
# -------------------------------------------------------
# Extract State/UT Total rows
# -------------------------------------------------------

state_total_raw = density_raw[
    (density_raw[3] == "STATE") &
    (density_raw[5] == "Total")
].copy()

print("Shape:", state_total_raw.shape)

print("\nExtracted State/UT rows:")

display(
    state_total_raw[
        [0, 3, 4, 5, 10, 13, 14]
    ].reset_index(drop=True)
)

Shape: (35, 15)

Extracted State/UT rows:


,0,3,4,5,10,13,14
0,01,STATE,JAMMU & KASHMIR @&,Total,12541302,222236,124
1,02,STATE,HIMACHAL PRADESH,Total,6864602,55673,123
2,03,STATE,PUNJAB,Total,27743338,50362,551
3,04,STATE,CHANDIGARH,Total,1055450,114,9258
4,05,STATE,UTTARAKHAND,Total,10086292,53483,189
5,06,STATE,HARYANA,Total,25351462,44212,573
6,07,STATE,NCT OF DELHI,Total,16787941,1483,11320
7,08,STATE,RAJASTHAN,Total,68548437,342239,200
8,09,STATE,UTTAR PRADESH,Total,199812341,240928,829
9,10,STATE,BIHAR,Total,104099452,94163,1106


In [8]:
# -------------------------------------------------------
# Validate extracted State/UT rows
# -------------------------------------------------------

print("Number of State/UT rows:", len(state_total_raw))

print(
    "\nUnique State/UT names:",
    state_total_raw[4].nunique()
)

print(
    "\nDuplicate names:"
)

duplicates = (
    state_total_raw[4]
    .value_counts()
    .loc[lambda x: x > 1]
)

display(duplicates)

print(
    "\nAdministrative levels present:",
    state_total_raw[3].unique()
)

print(
    "\nCategories present:",
    state_total_raw[5].unique()
)

Number of State/UT rows: 35

Unique State/UT names: 35

Duplicate names:


Series([], Name: count, dtype: int64)


Administrative levels present: ['STATE']

Categories present: ['Total']


In [9]:
# -------------------------------------------------------
# Create clean Census 2011 state-level dataset
# -------------------------------------------------------

density_df = state_total_raw[
    [4, 10, 13, 14]
].copy()

density_df.columns = [
    "source_state",
    "population_2011",
    "area_sq_km_2011",
    "source_population_density"
]

# Convert numeric columns explicitly
density_df["population_2011"] = pd.to_numeric(
    density_df["population_2011"],
    errors="coerce"
)

density_df["area_sq_km_2011"] = pd.to_numeric(
    density_df["area_sq_km_2011"],
    errors="coerce"
)

density_df["source_population_density"] = pd.to_numeric(
    density_df["source_population_density"],
    errors="coerce"
)

print("Shape:", density_df.shape)

display(density_df.head())

Shape: (35, 4)


,source_state,population_2011,area_sq_km_2011,source_population_density
7,JAMMU & KASHMIR @&,12541302,222236,124.0
322,HIMACHAL PRADESH,6864602,55673,123.0
712,PUNJAB,27743338,50362,551.0
1006,CHANDIGARH,1055450,114,9258.0
1015,UTTARAKHAND,10086292,53483,189.0


In [10]:
# -------------------------------------------------------
# Standardize Census state/UT names
# -------------------------------------------------------

name_mapping = {
    "JAMMU & KASHMIR @&": "Jammu & Kashmir",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "PUNJAB": "Punjab",
    "CHANDIGARH": "Chandigarh",
    "UTTARAKHAND": "Uttarakhand",
    "HARYANA": "Haryana",
    "NCT OF DELHI": "N.C.T of Delhi",
    "RAJASTHAN": "Rajasthan",
    "UTTAR PRADESH": "Uttar Pradesh",
    "BIHAR": "Bihar",
    "SIKKIM": "Sikkim",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "NAGALAND": "Nagaland",
    "MANIPUR": "Manipur",
    "MIZORAM": "Mizoram",
    "TRIPURA": "Tripura",
    "MEGHALAYA": "Meghalaya",
    "ASSAM": "Assam",
    "WEST BENGAL": "West Bengal",
    "JHARKHAND": "Jharkhand",
    "ODISHA": "Odisha",
    "CHHATTISGARH": "Chhattisgarh",
    "MADHYA PRADESH": "Madhya Pradesh",
    "GUJARAT": "Gujarat",
    "MAHARASHTRA": "Maharashtra",
    "ANDHRA PRADESH": "Andhra Pradesh",
    "KARNATAKA": "Karnataka",
    "GOA": "Goa",
    "LAKSHADWEEP": "Lakshadweep",
    "KERALA": "Kerala",
    "TAMIL NADU": "Tamil Nadu",
    "PUDUCHERRY": "Puducherry",
    "ANDAMAN & NICOBAR ISLANDS": "Andaman & Nicobar Islands"
}

density_df["state"] = density_df["source_state"].map(name_mapping)

# Check unmapped names
unmapped = density_df[
    density_df["state"].isna()
]["source_state"].unique()

print("Unmapped names:", unmapped)

Unmapped names: ['DAMAN & DIU' 'DADRA & NAGAR HAVELI']


In [11]:
# -------------------------------------------------------
# Combine Dadra & Nagar Haveli + Daman & Diu
# -------------------------------------------------------

old_ut_mask = density_df["source_state"].isin([
    "DAMAN & DIU",
    "DADRA & NAGAR HAVELI"
])

combined_ut = density_df.loc[
    old_ut_mask,
    [
        "population_2011",
        "area_sq_km_2011"
    ]
].sum()

combined_row = pd.DataFrame({
    "source_state": [
        "Dadra & Nagar Haveli + Daman & Diu"
    ],
    "population_2011": [
        combined_ut["population_2011"]
    ],
    "area_sq_km_2011": [
        combined_ut["area_sq_km_2011"]
    ],
    "source_population_density": [
        combined_ut["population_2011"] /
        combined_ut["area_sq_km_2011"]
    ],
    "state": [
        "Dadra & Nagar Haveli and Daman & Diu"
    ]
})

density_df = pd.concat(
    [
        density_df.loc[~old_ut_mask],
        combined_row
    ],
    ignore_index=True
)

print("Rows after combining old UTs:", len(density_df))

display(
    density_df[
        [
            "state",
            "population_2011",
            "area_sq_km_2011"
        ]
    ]
)

Rows after combining old UTs: 34


,state,population_2011,area_sq_km_2011
0,Jammu & Kashmir,12541302,222236
1,Himachal Pradesh,6864602,55673
2,Punjab,27743338,50362
3,Chandigarh,1055450,114
4,Uttarakhand,10086292,53483
5,Haryana,25351462,44212
6,N.C.T of Delhi,16787941,1483
7,Rajasthan,68548437,342239
8,Uttar Pradesh,199812341,240928
9,Bihar,104099452,94163


In [12]:
# -------------------------------------------------------
# Compare Census coverage with RoadSafe master dataset
# -------------------------------------------------------

master_path = (
    "../data/processed/master_state_analysis_2024.csv"
)

master_df = pd.read_csv(master_path)

print("RoadSafe states:", len(master_df))
print("Census states:", len(density_df))

road_states = set(master_df["state"])
census_states = set(density_df["state"])

missing_in_census = sorted(
    road_states - census_states
)

extra_in_census = sorted(
    census_states - road_states
)

print("\nStates present in RoadSafe but missing in Census:")
print(missing_in_census)

print("\nStates present in Census but not in RoadSafe:")
print(extra_in_census)

RoadSafe states: 36
Census states: 34

States present in RoadSafe but missing in Census:
['Ladakh', 'Telangana']

States present in Census but not in RoadSafe:
[]


In [13]:
# -------------------------------------------------------
# Find Telangana in the raw Census 2011 data
# -------------------------------------------------------

telangana_rows = density_raw[
    density_raw[4]
    .astype(str)
    .str.upper()
    .str.contains("TELANG", na=False)
]

print("Rows containing Telangana:")
display(telangana_rows[[0, 3, 4, 5, 10, 13, 14]])

Rows containing Telangana:


,0,3,4,5,10,13,14


In [14]:
# -------------------------------------------------------
# Check Census state list and possible alternate names
# -------------------------------------------------------

print("All extracted Census State/UT names:")
print()

for state in density_df["source_state"].tolist():
    print(state)

print("\nRows containing possible Telangana-related terms:")
possible_telangana = density_raw[
    density_raw[4]
    .astype(str)
    .str.upper()
    .str.contains(
        "TELANG|ANDHRA",
        na=False
    )
]

display(
    possible_telangana[
        [0, 3, 4, 5, 10, 13, 14]
    ]
)

All extracted Census State/UT names:

JAMMU & KASHMIR @&
HIMACHAL PRADESH
PUNJAB
CHANDIGARH
UTTARAKHAND
HARYANA
NCT OF DELHI
RAJASTHAN
UTTAR PRADESH
BIHAR
SIKKIM
ARUNACHAL PRADESH
NAGALAND
MANIPUR
MIZORAM
TRIPURA
MEGHALAYA
ASSAM
WEST BENGAL
JHARKHAND
ODISHA
CHHATTISGARH
MADHYA PRADESH
GUJARAT
MAHARASHTRA
ANDHRA PRADESH
KARNATAKA
GOA
LAKSHADWEEP
KERALA
TAMIL NADU
PUDUCHERRY
ANDAMAN & NICOBAR ISLANDS
Dadra & Nagar Haveli + Daman & Diu

Rows containing possible Telangana-related terms:


,0,3,4,5,10,13,14
3943,10,SUB-DISTRICT,Andhratharhi,Total,191680,145.320362,1319
3944,10,SUB-DISTRICT,Andhratharhi,Rural,191680,145.320362,1319
3945,10,SUB-DISTRICT,Andhratharhi,Urban,0,0,0
14770,28,STATE,ANDHRA PRADESH,Total,84580777,275045,307.516141
14771,28,STATE,ANDHRA PRADESH,Rural,56361702,267190.5,210.942013
14772,28,STATE,ANDHRA PRADESH,Urban,28219075,7854.5,3592.727099


In [15]:
# -------------------------------------------------------
# Calculate Population Density 2011
# -------------------------------------------------------

density_df["population_density_2011"] = (
    density_df["population_2011"]
    / density_df["area_sq_km_2011"]
)

# Compare our calculation with the source-provided density
density_df["density_difference"] = (
    density_df["population_density_2011"]
    - density_df["source_population_density"]
)

display(
    density_df[
        [
            "state",
            "population_2011",
            "area_sq_km_2011",
            "source_population_density",
            "population_density_2011",
            "density_difference"
        ]
    ].head(10)
)

,state,population_2011,area_sq_km_2011,source_population_density,population_density_2011,density_difference
0,Jammu & Kashmir,12541302,222236,124.0,56.432360,-67.567640
1,Himachal Pradesh,6864602,55673,123.0,123.302175,0.302175
2,Punjab,27743338,50362,551.0,550.878400,-0.121600
3,Chandigarh,1055450,114,9258.0,9258.333333,0.333333
4,Uttarakhand,10086292,53483,189.0,188.588748,-0.411252
5,Haryana,25351462,44212,573.0,573.406813,0.406813
6,N.C.T of Delhi,16787941,1483,11320.0,11320.256912,0.256912
7,Rajasthan,68548437,342239,200.0,200.294055,0.294055
8,Uttar Pradesh,199812341,240928,829.0,829.344622,0.344622
9,Bihar,104099452,94163,1106.0,1105.523953,-0.476047


In [16]:
# -------------------------------------------------------
# Validate population density calculation
# -------------------------------------------------------

max_difference = density_df["density_difference"].abs().max()

print(
    "Maximum absolute difference from source density:",
    max_difference
)

print(
    "\nMissing population values:",
    density_df["population_2011"].isna().sum()
)

print(
    "Missing area values:",
    density_df["area_sq_km_2011"].isna().sum()
)

print(
    "Missing calculated density:",
    density_df["population_density_2011"].isna().sum()
)

Maximum absolute difference from source density: 67.56763980633201

Missing population values: 0
Missing area values: 0
Missing calculated density: 0


In [17]:
# -------------------------------------------------------
# Create final research-ready population density dataset
# -------------------------------------------------------

density_final = density_df[
    [
        "state",
        "population_2011",
        "area_sq_km_2011",
        "population_density_2011"
    ]
].copy()

# Sort alphabetically
density_final = density_final.sort_values(
    "state"
).reset_index(drop=True)

# Save
density_output_path = (
    "../data/research/"
    "population_density_2011_state_level.csv"
)

density_final.to_csv(
    density_output_path,
    index=False
)

print(
    "Saved:",
    density_output_path
)

print(
    "\nShape:",
    density_final.shape
)

display(density_final)

Saved: ../data/research/population_density_2011_state_level.csv

Shape: (34, 4)


,state,population_2011,area_sq_km_2011,population_density_2011
0,Andaman & Nicobar Islands,380581,8249,46.136623
1,Andhra Pradesh,84580777,275045,307.516141
2,Arunachal Pradesh,1383727,83743,16.523495
3,Assam,31205576,78438,397.837477
4,Bihar,104099452,94163,1105.523953
5,Chandigarh,1055450,114,9258.333333
6,Chhattisgarh,25545198,135192,188.954953
7,Dadra & Nagar Haveli and Daman & Diu,586956,602,975.009967
8,Goa,1458545,3702,393.988385
9,Gujarat,60439692,196244,307.982369


In [18]:
# -------------------------------------------------------
# Final coverage check
# -------------------------------------------------------

road_states = set(master_df["state"])
density_states = set(density_final["state"])

missing_states = sorted(
    road_states - density_states
)

extra_states = sorted(
    density_states - road_states
)

print("RoadSafe states:", len(road_states))
print("Density states:", len(density_states))

print(
    "\nMissing from Census density dataset:"
)

print(missing_states)

print(
    "\nExtra Census states:"
)

print(extra_states)

RoadSafe states: 36
Density states: 34

Missing from Census density dataset:
['Ladakh', 'Telangana']

Extra Census states:
[]


In [19]:
# -------------------------------------------------------
# Load RoadSafe datasets for integration
# -------------------------------------------------------

master_path = (
    "../data/processed/"
    "master_state_analysis_2024.csv"
)

vehicle_path = (
    "../data/research/"
    "vehicle_exposure_2024_state_level.csv"
)

road_path = (
    "../data/research/"
    "road_infrastructure_2019_state_level.csv"
)

master_df = pd.read_csv(master_path)
vehicle_df = pd.read_csv(vehicle_path)
road_df = pd.read_csv(road_path)

print("Master dataset:", master_df.shape)
print("Vehicle dataset:", vehicle_df.shape)
print("Road dataset:", road_df.shape)
print("Density dataset:", density_final.shape)

Master dataset: (36, 26)
Vehicle dataset: (36, 28)
Road dataset: (36, 30)
Density dataset: (34, 4)


In [20]:
# -------------------------------------------------------
# Inspect columns before merging
# -------------------------------------------------------

print("MASTER:")
print(master_df.columns.tolist())

print("\nVEHICLE:")
print(vehicle_df.columns.tolist())

print("\nROAD:")
print(road_df.columns.tolist())

print("\nDENSITY:")
print(density_final.columns.tolist())

MASTER:
['state', '2020_accidents', '2021_accidents', '2022_accidents', '2023_accidents', '2024_accidents', '2020_killed', '2021_killed', '2022_killed', '2023_killed', '2024_killed', 'population_2024', 'accidents_per_100k_population', 'fatalities_per_100k_population', 'fatalities_per_100_accidents', 'five_year_average_accidents', 'five_year_average_fatalities', 'fatality_years_available', 'accident_change_2020_to_2024', 'accident_percent_change_2020_to_2024', 'fatality_change_2020_to_2024', 'fatality_percent_change_2020_to_2024', 'accident_rank_2024', 'fatality_rank_2024', 'accident_rate_rank_2024', 'fatality_rate_rank_2024']

VEHICLE:
['state', '2020_accidents', '2021_accidents', '2022_accidents', '2023_accidents', '2024_accidents', '2020_killed', '2021_killed', '2022_killed', '2023_killed', '2024_killed', 'population_2024', 'accidents_per_100k_population', 'fatalities_per_100k_population', 'fatalities_per_100_accidents', 'five_year_average_accidents', 'five_year_average_fatalities', 

In [21]:
# -------------------------------------------------------
# Build integrated research dataset
# -------------------------------------------------------

# Keep only the new variables from each research dataset

vehicle_research = vehicle_df[
    [
        "state",
        "vehicle_registrations_2024",
        "vehicles_per_1000_population"
    ]
].copy()

road_research = road_df[
    [
        "state",
        "road_length_2019_km",
        "road_length_per_1000_population"
    ]
].copy()

density_research = density_final[
    [
        "state",
        "population_2011",
        "area_sq_km_2011",
        "population_density_2011"
    ]
].copy()


# -------------------------------------------------------
# Start with the complete RoadSafe 2024 master
# -------------------------------------------------------

research_df = master_df.copy()


# Add vehicle exposure
research_df = research_df.merge(
    vehicle_research,
    on="state",
    how="left",
    validate="one_to_one"
)


# Add road infrastructure
research_df = research_df.merge(
    road_research,
    on="state",
    how="left",
    validate="one_to_one"
)


# Add population density
research_df = research_df.merge(
    density_research,
    on="state",
    how="left",
    validate="one_to_one"
)


print("Integrated dataset shape:", research_df.shape)

display(
    research_df[
        [
            "state",
            "2024_accidents",
            "2024_killed",
            "population_2024",
            "vehicle_registrations_2024",
            "vehicles_per_1000_population",
            "road_length_2019_km",
            "road_length_per_1000_population",
            "population_2011",
            "area_sq_km_2011",
            "population_density_2011"
        ]
    ].head(10)
)

Integrated dataset shape: (36, 33)


,state,2024_accidents,2024_killed,population_2024,vehicle_registrations_2024,vehicles_per_1000_population,road_length_2019_km,road_length_per_1000_population,population_2011,area_sq_km_2011,population_density_2011
0,Andhra Pradesh,19557.0,8346,53340000,911747,17.093120,176351.0,3.306168,84580777.0,275045.0,307.516141
1,Arunachal Pradesh,277.0,168,1576000,36796,23.347716,55262.0,35.064721,1383727.0,83743.0,16.523495
2,Assam,7848.0,3351,36047000,629458,17.462147,399122.0,11.072267,31205576.0,78438.0,397.837477
3,Bihar,11610.0,9347,128592000,1395212,10.849913,298205.0,2.319001,104099452.0,94163.0,1105.523953
4,Chhattisgarh,14857.0,6945,30524000,714000,23.391430,105074.0,3.442340,25545198.0,135192.0,188.954953
5,Goa,2682.0,286,1583000,84300,53.253316,18697.0,11.811118,1458545.0,3702.0,393.988385
6,Gujarat,15588.0,7717,72367000,1899370,26.246355,249373.0,3.445949,60439692.0,196244.0,307.982369
7,Haryana,9806.0,4689,30573000,996158,32.582933,50292.0,1.644981,25351462.0,44212.0,573.406813
8,Himachal Pradesh,2156.0,869,7505000,147735,19.684877,73230.0,9.757495,6864602.0,55673.0,123.302175
9,Jharkhand,5196.0,4114,39963000,579217,14.493832,81245.0,2.033006,32988134.0,79716.0,413.820739


In [22]:
# -------------------------------------------------------
# Validate integrated research dataset
# -------------------------------------------------------

print("Number of states:", research_df["state"].nunique())

print(
    "\nDuplicate states:",
    research_df["state"].duplicated().sum()
)

print("\nMissing values in research variables:")

research_variables = [
    "vehicle_registrations_2024",
    "vehicles_per_1000_population",
    "road_length_2019_km",
    "road_length_per_1000_population",
    "population_2011",
    "area_sq_km_2011",
    "population_density_2011"
]

display(
    research_df[
        ["state"] + research_variables
    ].isna().sum()
)

print("\nStates with missing research variables:")

display(
    research_df[
        ["state"] + research_variables
    ][
        research_df[research_variables]
        .isna()
        .any(axis=1)
    ]
)

Number of states: 36

Duplicate states: 0

Missing values in research variables:


state                              0
vehicle_registrations_2024         0
vehicles_per_1000_population       0
road_length_2019_km                1
road_length_per_1000_population    1
population_2011                    2
area_sq_km_2011                    2
population_density_2011            2
dtype: int64


States with missing research variables:


,state,vehicle_registrations_2024,vehicles_per_1000_population,road_length_2019_km,road_length_per_1000_population,population_2011,area_sq_km_2011,population_density_2011
23,Telangana,1026491,26.820940,140555.0,3.672528,NaN,NaN,NaN
33,Ladakh,5437,18.003311,NaN,NaN,NaN,NaN,NaN


In [23]:
# -------------------------------------------------------
# Save integrated research dataset
# -------------------------------------------------------

integrated_output_path = (
    "../data/research/"
    "integrated_research_dataset_2024.csv"
)

research_df.to_csv(
    integrated_output_path,
    index=False
)

print(
    "Saved:",
    integrated_output_path
)

print(
    "Shape:",
    research_df.shape
)

Saved: ../data/research/integrated_research_dataset_2024.csv
Shape: (36, 33)


In [24]:
# -------------------------------------------------------
# Create complete-case dataset for 3-variable analysis
# -------------------------------------------------------

predictors = [
    "vehicles_per_1000_population",
    "road_length_per_1000_population",
    "population_density_2011"
]

outcomes = [
    "accidents_per_100k_population",
    "fatalities_per_100k_population"
]

regression_3var_df = research_df.dropna(
    subset=predictors + outcomes
).copy()

print(
    "Regression-ready observations:",
    len(regression_3var_df)
)

print(
    "\nExcluded states:"
)

excluded_states = sorted(
    set(research_df["state"])
    - set(regression_3var_df["state"])
)

print(excluded_states)

display(
    regression_3var_df[
        ["state"] + predictors + outcomes
    ]
)

Regression-ready observations: 34

Excluded states:
['Ladakh', 'Telangana']


,state,vehicles_per_1000_population,road_length_per_1000_population,population_density_2011,accidents_per_100k_population,fatalities_per_100k_population
0,Andhra Pradesh,17.093120,3.306168,307.516141,36.664792,15.646794
1,Arunachal Pradesh,23.347716,35.064721,16.523495,17.576142,10.659898
2,Assam,17.462147,11.072267,397.837477,21.771576,9.296197
3,Bihar,10.849913,2.319001,1105.523953,9.028555,7.268726
4,Chhattisgarh,23.391430,3.442340,188.954953,48.673175,22.752588
5,Goa,53.253316,11.811118,393.988385,169.425142,18.066961
6,Gujarat,26.246355,3.445949,307.982369,21.540205,10.663700
7,Haryana,32.582933,1.644981,573.406813,32.074052,15.337062
8,Himachal Pradesh,19.684877,9.757495,123.302175,28.727515,11.578947
9,Jharkhand,14.493832,2.033006,413.820739,13.002027,10.294522


In [25]:
# -------------------------------------------------------
# Correlation analysis among regression predictors
# -------------------------------------------------------

predictor_df = regression_3var_df[
    [
        "vehicles_per_1000_population",
        "road_length_per_1000_population",
        "population_density_2011"
    ]
].copy()

print("Pearson correlation matrix:")

display(
    predictor_df.corr(method="pearson").round(4)
)

print("\nSpearman correlation matrix:")

display(
    predictor_df.corr(method="spearman").round(4)
)

Pearson correlation matrix:


,vehicles_per_1000_population,road_length_per_1000_population,population_density_2011
vehicles_per_1000_population,1.0000,-0.0729,0.3692
road_length_per_1000_population,-0.0729,1.0000,-0.3008
population_density_2011,0.3692,-0.3008,1.0000



Spearman correlation matrix:


,vehicles_per_1000_population,road_length_per_1000_population,population_density_2011
vehicles_per_1000_population,1.0000,-0.1349,0.1487
road_length_per_1000_population,-0.1349,1.0000,-0.7268
population_density_2011,0.1487,-0.7268,1.0000


In [26]:
# -------------------------------------------------------
# Variance Inflation Factor (VIF)
# -------------------------------------------------------

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

X_vif = predictor_df.copy()

# Add intercept
X_vif_with_constant = X_vif.copy()
X_vif_with_constant.insert(
    0,
    "const",
    1.0
)

vif_results = []

for i, column in enumerate(
    X_vif_with_constant.columns
):
    if column == "const":
        continue

    vif_results.append({
        "variable": column,
        "VIF": variance_inflation_factor(
            X_vif_with_constant.values,
            i
        )
    })

vif_df = pd.DataFrame(vif_results)

display(
    vif_df.round(4)
)

,variable,VIF
0,vehicles_per_1000_population,1.1599
1,road_length_per_1000_population,1.1015
2,population_density_2011,1.2685


In [27]:
# -------------------------------------------------------
# Multicollinearity assessment
# -------------------------------------------------------

print("Multicollinearity assessment")
print("=" * 50)

for _, row in vif_df.iterrows():

    variable = row["variable"]
    vif = row["VIF"]

    if vif < 5:
        assessment = "Low multicollinearity"
    elif vif < 10:
        assessment = "Moderate multicollinearity"
    else:
        assessment = "High multicollinearity"

    print(
        f"{variable}: VIF = {vif:.3f} → {assessment}"
    )

Multicollinearity assessment
vehicles_per_1000_population: VIF = 1.160 → Low multicollinearity
road_length_per_1000_population: VIF = 1.102 → Low multicollinearity
population_density_2011: VIF = 1.269 → Low multicollinearity


In [28]:
# -------------------------------------------------------
# 3-Predictor Multivariable Regression
# -------------------------------------------------------

import statsmodels.api as sm

predictor_columns = [
    "vehicles_per_1000_population",
    "road_length_per_1000_population",
    "population_density_2011"
]

# -------------------------------------------------------
# Prepare predictors
# -------------------------------------------------------

X = regression_3var_df[
    predictor_columns
].copy()

X = sm.add_constant(X)

# -------------------------------------------------------
# Outcomes
# -------------------------------------------------------

y_accidents = regression_3var_df[
    "accidents_per_100k_population"
]

y_fatalities = regression_3var_df[
    "fatalities_per_100k_population"
]

# -------------------------------------------------------
# Accident-rate model
# -------------------------------------------------------

accident_model_3var = sm.OLS(
    y_accidents,
    X
).fit(
    cov_type="HC3"
)

# -------------------------------------------------------
# Fatality-rate model
# -------------------------------------------------------

fatality_model_3var = sm.OLS(
    y_fatalities,
    X
).fit(
    cov_type="HC3"
)


# -------------------------------------------------------
# Display robust coefficient results
# -------------------------------------------------------

def robust_results_table(model):
    return pd.DataFrame({
        "Coefficient": model.params,
        "Robust_SE": model.bse,
        "t_value": model.tvalues,
        "p_value": model.pvalues
    })


print("ACCIDENT RATE MODEL")
print("=" * 70)

display(
    robust_results_table(
        accident_model_3var
    ).round(4)
)

print(
    f"R² = {accident_model_3var.rsquared:.4f}"
)

print(
    f"Adjusted R² = "
    f"{accident_model_3var.rsquared_adj:.4f}"
)


print("\nFATALITY RATE MODEL")
print("=" * 70)

display(
    robust_results_table(
        fatality_model_3var
    ).round(4)
)

print(
    f"R² = {fatality_model_3var.rsquared:.4f}"
)

print(
    f"Adjusted R² = "
    f"{fatality_model_3var.rsquared_adj:.4f}"
)

ACCIDENT RATE MODEL


,Coefficient,Robust_SE,t_value,p_value
const,-18.5255,19.3482,-0.9575,0.3383
vehicles_per_1000_population,2.9705,1.0025,2.9630,0.0030
road_length_per_1000_population,-0.5668,0.7690,-0.7371,0.4611
population_density_2011,-0.0058,0.0030,-1.9271,0.0540


R² = 0.5036
Adjusted R² = 0.4539

FATALITY RATE MODEL


,Coefficient,Robust_SE,t_value,p_value
const,5.2924,3.0043,1.7616,0.0781
vehicles_per_1000_population,0.3893,0.1343,2.8979,0.0038
road_length_per_1000_population,-0.2242,0.2198,-1.0202,0.3076
population_density_2011,-0.0012,0.0005,-2.3100,0.0209


R² = 0.4844
Adjusted R² = 0.4328


In [29]:
# -------------------------------------------------------
# Compare 2-predictor vs 3-predictor models
# using the SAME 34-state sample
# -------------------------------------------------------

# -------------------------------------------------------
# 2-predictor specification
# -------------------------------------------------------

two_predictors = [
    "vehicles_per_1000_population",
    "road_length_per_1000_population"
]

three_predictors = [
    "vehicles_per_1000_population",
    "road_length_per_1000_population",
    "population_density_2011"
]


# -------------------------------------------------------
# Prepare 2-predictor model on the SAME 34 observations
# -------------------------------------------------------

X_2 = regression_3var_df[
    two_predictors
].copy()

X_2 = sm.add_constant(X_2)

X_3 = regression_3var_df[
    three_predictors
].copy()

X_3 = sm.add_constant(X_3)


# -------------------------------------------------------
# Outcomes
# -------------------------------------------------------

y_acc = regression_3var_df[
    "accidents_per_100k_population"
]

y_fat = regression_3var_df[
    "fatalities_per_100k_population"
]


# -------------------------------------------------------
# Fit models with ordinary OLS
# -------------------------------------------------------
# We use ordinary OLS here for the nested-model comparison.
# HC3 robust inference remains our preferred coefficient
# inference for the substantive results.

acc_2_ols = sm.OLS(y_acc, X_2).fit()
acc_3_ols = sm.OLS(y_acc, X_3).fit()

fat_2_ols = sm.OLS(y_fat, X_2).fit()
fat_3_ols = sm.OLS(y_fat, X_3).fit()


# -------------------------------------------------------
# Fit robust versions as well
# -------------------------------------------------------

acc_2_hc3 = sm.OLS(
    y_acc,
    X_2
).fit(cov_type="HC3")

acc_3_hc3 = sm.OLS(
    y_acc,
    X_3
).fit(cov_type="HC3")

fat_2_hc3 = sm.OLS(
    y_fat,
    X_2
).fit(cov_type="HC3")

fat_3_hc3 = sm.OLS(
    y_fat,
    X_3
).fit(cov_type="HC3")


# -------------------------------------------------------
# Comparison table
# -------------------------------------------------------

comparison = pd.DataFrame({
    "Model": [
        "Accident - 2 predictors",
        "Accident - 3 predictors",
        "Fatality - 2 predictors",
        "Fatality - 3 predictors"
    ],
    "N": [
        len(y_acc),
        len(y_acc),
        len(y_fat),
        len(y_fat)
    ],
    "R2": [
        acc_2_ols.rsquared,
        acc_3_ols.rsquared,
        fat_2_ols.rsquared,
        fat_3_ols.rsquared
    ],
    "Adjusted_R2": [
        acc_2_ols.rsquared_adj,
        acc_3_ols.rsquared_adj,
        fat_2_ols.rsquared_adj,
        fat_3_ols.rsquared_adj
    ],
    "AIC": [
        acc_2_ols.aic,
        acc_3_ols.aic,
        fat_2_ols.aic,
        fat_3_ols.aic
    ],
    "BIC": [
        acc_2_ols.bic,
        acc_3_ols.bic,
        fat_2_ols.bic,
        fat_3_ols.bic
    ]
})

display(
    comparison.round(4)
)


# -------------------------------------------------------
# Nested-model F tests
# -------------------------------------------------------

print("ACCIDENT MODEL — ADDING POPULATION DENSITY")
print("=" * 60)

display(
    acc_2_ols.compare_f_test(
        acc_3_ols
    )
)

print("\nFATALITY MODEL — ADDING POPULATION DENSITY")
print("=" * 60)

display(
    fat_2_ols.compare_f_test(
        fat_3_ols
    )
)


# -------------------------------------------------------
# Robust p-values for population density
# -------------------------------------------------------

print("\nHC3 ROBUST p-values")

print(
    "Accident model — population density:",
    acc_3_hc3.pvalues[
        "population_density_2011"
    ]
)

print(
    "Fatality model — population density:",
    fat_3_hc3.pvalues[
        "population_density_2011"
    ]
)

,Model,N,R2,Adjusted_R2,AIC,BIC
0,Accident - 2 predictors,34,0.3909,0.3516,329.7548,334.3339
1,Accident - 3 predictors,34,0.5036,0.4539,324.8033,330.9087
2,Fatality - 2 predictors,34,0.2512,0.2029,207.2569,211.8360
3,Fatality - 3 predictors,34,0.4844,0.4328,196.5708,202.6762


ACCIDENT MODEL — ADDING POPULATION DENSITY


(np.float64(5.7321871934748145), np.float64(nan), np.float64(-1.0))


FATALITY MODEL — ADDING POPULATION DENSITY


(np.float64(9.65397793925733), np.float64(nan), np.float64(-1.0))


HC3 ROBUST p-values
Accident model — population density: 0.0539667056429866
Fatality model — population density: 0.020887474738200938


In [30]:
# -------------------------------------------------------
# Correct nested-model F tests
# -------------------------------------------------------

print("ACCIDENT MODEL — ADDING POPULATION DENSITY")
print("=" * 60)

accident_f_test = acc_3_ols.compare_f_test(
    acc_2_ols
)

print(
    "F-statistic:",
    accident_f_test[0]
)

print(
    "p-value:",
    accident_f_test[1]
)

print(
    "Degrees of freedom difference:",
    accident_f_test[2]
)


print("\nFATALITY MODEL — ADDING POPULATION DENSITY")
print("=" * 60)

fatality_f_test = fat_3_ols.compare_f_test(
    fat_2_ols
)

print(
    "F-statistic:",
    fatality_f_test[0]
)

print(
    "p-value:",
    fatality_f_test[1]
)

print(
    "Degrees of freedom difference:",
    fatality_f_test[2]
)

ACCIDENT MODEL — ADDING POPULATION DENSITY
F-statistic: 6.8057182915268335
p-value: 0.01403247966122887
Degrees of freedom difference: 1.0

FATALITY MODEL — ADDING POPULATION DENSITY
F-statistic: 13.56783654366951
p-value: 0.0009040385170720012
Degrees of freedom difference: 1.0


In [32]:
# -------------------------------------------------------
# Residual normality diagnostics for 3-predictor models
# -------------------------------------------------------

from statsmodels.stats.stattools import omni_normtest, jarque_bera

# -------------------------------------------------------
# Omnibus test
# -------------------------------------------------------

acc_omni = omni_normtest(
    acc_3_ols.resid
)

fat_omni = omni_normtest(
    fat_3_ols.resid
)

print("RESIDUAL NORMALITY — OMNIBUS TEST")
print("=" * 60)

print(
    f"Accident model: statistic = "
    f"{acc_omni[0]:.4f}, p = {acc_omni[1]:.6f}"
)

print(
    f"Fatality model: statistic = "
    f"{fat_omni[0]:.4f}, p = {fat_omni[1]:.6f}"
)


# -------------------------------------------------------
# Jarque-Bera test
# -------------------------------------------------------

acc_jb = jarque_bera(
    acc_3_ols.resid
)

fat_jb = jarque_bera(
    fat_3_ols.resid
)

print("\nJARQUE-BERA TEST")
print("=" * 60)

print(
    f"Accident model: statistic = "
    f"{acc_jb[0]:.4f}, p = {acc_jb[1]:.6f}"
)

print(
    f"Fatality model: statistic = "
    f"{fat_jb[0]:.4f}, p = {fat_jb[1]:.6f}"
)

RESIDUAL NORMALITY — OMNIBUS TEST
Accident model: statistic = 23.0722, p = 0.000010
Fatality model: statistic = 4.8357, p = 0.089114

JARQUE-BERA TEST
Accident model: statistic = 44.1622, p = 0.000000
Fatality model: statistic = 3.8299, p = 0.147350


In [33]:
# -------------------------------------------------------
# Cook's Distance — 3-predictor models
# -------------------------------------------------------

import numpy as np
import pandas as pd

# -------------------------------------------------------
# Accident model influence
# -------------------------------------------------------

acc_influence = acc_3_ols.get_influence()
acc_cooks = acc_influence.cooks_distance[0]

acc_cook_df = pd.DataFrame({
    "state": regression_3var_df["state"].values,
    "cooks_distance": acc_cooks
}).sort_values(
    "cooks_distance",
    ascending=False
).reset_index(drop=True)


# -------------------------------------------------------
# Fatality model influence
# -------------------------------------------------------

fat_influence = fat_3_ols.get_influence()
fat_cooks = fat_influence.cooks_distance[0]

fat_cook_df = pd.DataFrame({
    "state": regression_3var_df["state"].values,
    "cooks_distance": fat_cooks
}).sort_values(
    "cooks_distance",
    ascending=False
).reset_index(drop=True)


# -------------------------------------------------------
# Display
# -------------------------------------------------------

print("ACCIDENT RATE MODEL — COOK'S DISTANCE")
print("=" * 60)

display(
    acc_cook_df.head(10).round(4)
)

print("\nFATALITY RATE MODEL — COOK'S DISTANCE")
print("=" * 60)

display(
    fat_cook_df.head(10).round(4)
)


# -------------------------------------------------------
# Common reference threshold
# -------------------------------------------------------

threshold = 4 / len(regression_3var_df)

print(
    f"\nCommon Cook's distance reference threshold "
    f"(4/n): {threshold:.4f}"
)

print("\nAccident observations above threshold:")

display(
    acc_cook_df[
        acc_cook_df["cooks_distance"] > threshold
    ].round(4)
)

print("\nFatality observations above threshold:")

display(
    fat_cook_df[
        fat_cook_df["cooks_distance"] > threshold
    ].round(4)
)

ACCIDENT RATE MODEL — COOK'S DISTANCE


,state,cooks_distance
0,Goa,0.8509
1,Arunachal Pradesh,0.2096
2,Chandigarh,0.2057
3,N.C.T of Delhi,0.1899
4,Kerala,0.1052
5,Haryana,0.0883
6,Mizoram,0.0294
7,Gujarat,0.0274
8,Tamil Nadu,0.0255
9,Uttarakhand,0.0175



FATALITY RATE MODEL — COOK'S DISTANCE


,state,cooks_distance
0,Arunachal Pradesh,0.8938
1,Goa,0.5738
2,N.C.T of Delhi,0.5112
3,Tamil Nadu,0.0894
4,Chhattisgarh,0.0750
5,Lakshadweep,0.0627
6,Chandigarh,0.0556
7,Puducherry,0.0418
8,Nagaland,0.0271
9,Manipur,0.0264



Common Cook's distance reference threshold (4/n): 0.1176

Accident observations above threshold:


,state,cooks_distance
0,Goa,0.8509
1,Arunachal Pradesh,0.2096
2,Chandigarh,0.2057
3,N.C.T of Delhi,0.1899



Fatality observations above threshold:


,state,cooks_distance
0,Arunachal Pradesh,0.8938
1,Goa,0.5738
2,N.C.T of Delhi,0.5112


In [34]:
# -------------------------------------------------------
# Sensitivity analysis for influential observations
# -------------------------------------------------------

def fit_robust_models(data):

    X = data[
        [
            "vehicles_per_1000_population",
            "road_length_per_1000_population",
            "population_density_2011"
        ]
    ].copy()

    X = sm.add_constant(X)

    y_acc = data[
        "accidents_per_100k_population"
    ]

    y_fat = data[
        "fatalities_per_100k_population"
    ]

    acc_model = sm.OLS(
        y_acc,
        X
    ).fit(cov_type="HC3")

    fat_model = sm.OLS(
        y_fat,
        X
    ).fit(cov_type="HC3")

    return acc_model, fat_model


# -------------------------------------------------------
# Define sensitivity samples
# -------------------------------------------------------

samples = {
    "Full sample": [],
    "Without Goa": ["Goa"],
    "Without Arunachal Pradesh": ["Arunachal Pradesh"],
    "Without Goa + Arunachal Pradesh": [
        "Goa",
        "Arunachal Pradesh"
    ],
    "Without Delhi": ["N.C.T of Delhi"]
}


# -------------------------------------------------------
# Run models
# -------------------------------------------------------

results = []

for sample_name, excluded_states in samples.items():

    data = regression_3var_df[
        ~regression_3var_df["state"].isin(
            excluded_states
        )
    ].copy()

    acc_model, fat_model = fit_robust_models(
        data
    )

    results.append({
        "Sample": sample_name,
        "N": len(data),

        "Accident_vehicle_coef":
            acc_model.params[
                "vehicles_per_1000_population"
            ],

        "Accident_vehicle_p":
            acc_model.pvalues[
                "vehicles_per_1000_population"
            ],

        "Accident_road_p":
            acc_model.pvalues[
                "road_length_per_1000_population"
            ],

        "Accident_density_coef":
            acc_model.params[
                "population_density_2011"
            ],

        "Accident_density_p":
            acc_model.pvalues[
                "population_density_2011"
            ],

        "Accident_R2":
            acc_model.rsquared,

        "Fatality_vehicle_coef":
            fat_model.params[
                "vehicles_per_1000_population"
            ],

        "Fatality_vehicle_p":
            fat_model.pvalues[
                "vehicles_per_1000_population"
            ],

        "Fatality_road_p":
            fat_model.pvalues[
                "road_length_per_1000_population"
            ],

        "Fatality_density_coef":
            fat_model.params[
                "population_density_2011"
            ],

        "Fatality_density_p":
            fat_model.pvalues[
                "population_density_2011"
            ],

        "Fatality_R2":
            fat_model.rsquared
    })


sensitivity_df = pd.DataFrame(results)


# -------------------------------------------------------
# Display
# -------------------------------------------------------

display(
    sensitivity_df.round(4)
)

,Sample,N,Accident_vehicle_coef,Accident_vehicle_p,Accident_road_p,Accident_density_coef,Accident_density_p,Accident_R2,Fatality_vehicle_coef,Fatality_vehicle_p,Fatality_road_p,Fatality_density_coef,Fatality_density_p,Fatality_R2
0,Full sample,34,2.9705,0.0030,0.4611,-0.0058,0.0540,0.5036,0.3893,0.0038,0.3076,-0.0012,0.0209,0.4844
1,Without Goa,33,2.0174,0.0006,0.0291,-0.0043,0.0542,0.2556,0.5081,0.0001,0.2480,-0.0014,0.0171,0.5007
2,Without Arunachal Pradesh,33,2.9950,0.0026,0.9315,-0.0054,0.0592,0.5098,0.3816,0.0007,0.0136,-0.0013,0.0119,0.5271
3,Without Goa + Arunachal Pradesh,32,2.0492,0.0009,0.4294,-0.0042,0.0588,0.2513,0.4750,0.0003,0.0194,-0.0015,0.0147,0.5241
4,Without Delhi,33,3.0137,0.0023,0.4436,-0.0075,0.4695,0.5117,0.4001,0.0050,0.3055,-0.0017,0.3292,0.5029


In [35]:
# -------------------------------------------------------
# Log-outcome sensitivity models
# 3 predictors
# -------------------------------------------------------

# -------------------------------------------------------
# Remove zero-outcome observations for log transformation
# -------------------------------------------------------

log_df = regression_3var_df[
    (
        regression_3var_df[
            "accidents_per_100k_population"
        ] > 0
    )
    &
    (
        regression_3var_df[
            "fatalities_per_100k_population"
        ] > 0
    )
].copy()

print("Log-model observations:", len(log_df))

print(
    "Excluded states:",
    sorted(
        set(regression_3var_df["state"])
        - set(log_df["state"])
    )
)


# -------------------------------------------------------
# Create log outcomes
# -------------------------------------------------------

log_df["log_accident_rate"] = np.log(
    log_df["accidents_per_100k_population"]
)

log_df["log_fatality_rate"] = np.log(
    log_df["fatalities_per_100k_population"]
)


# -------------------------------------------------------
# Predictors
# -------------------------------------------------------

X_log = log_df[
    [
        "vehicles_per_1000_population",
        "road_length_per_1000_population",
        "population_density_2011"
    ]
].copy()

X_log = sm.add_constant(X_log)


# -------------------------------------------------------
# Fit HC3 robust models
# -------------------------------------------------------

log_accident_model = sm.OLS(
    log_df["log_accident_rate"],
    X_log
).fit(
    cov_type="HC3"
)

log_fatality_model = sm.OLS(
    log_df["log_fatality_rate"],
    X_log
).fit(
    cov_type="HC3"
)


# -------------------------------------------------------
# Results table
# -------------------------------------------------------

print("LOG ACCIDENT-RATE MODEL")
print("=" * 70)

display(
    pd.DataFrame({
        "Coefficient": log_accident_model.params,
        "Robust_SE": log_accident_model.bse,
        "t_value": log_accident_model.tvalues,
        "p_value": log_accident_model.pvalues
    }).round(4)
)

print(
    f"R² = {log_accident_model.rsquared:.4f}"
)

print(
    f"Adjusted R² = "
    f"{log_accident_model.rsquared_adj:.4f}"
)


print("\nLOG FATALITY-RATE MODEL")
print("=" * 70)

display(
    pd.DataFrame({
        "Coefficient": log_fatality_model.params,
        "Robust_SE": log_fatality_model.bse,
        "t_value": log_fatality_model.tvalues,
        "p_value": log_fatality_model.pvalues
    }).round(4)
)

print(
    f"R² = {log_fatality_model.rsquared:.4f}"
)

print(
    f"Adjusted R² = "
    f"{log_fatality_model.rsquared_adj:.4f}"
)

Log-model observations: 33
Excluded states: ['Lakshadweep']
LOG ACCIDENT-RATE MODEL


,Coefficient,Robust_SE,t_value,p_value
const,2.1713,0.2241,9.6881,0.0000
vehicles_per_1000_population,0.0633,0.0087,7.2504,0.0000
road_length_per_1000_population,-0.0273,0.0143,-1.9030,0.0570
population_density_2011,-0.0001,0.0001,-1.4726,0.1409


R² = 0.4631
Adjusted R² = 0.4076

LOG FATALITY-RATE MODEL


,Coefficient,Robust_SE,t_value,p_value
const,1.7578,0.3989,4.4066,0.0000
vehicles_per_1000_population,0.0372,0.0167,2.2238,0.0262
road_length_per_1000_population,-0.0241,0.0307,-0.7839,0.4331
population_density_2011,-0.0001,0.0001,-2.1257,0.0335


R² = 0.4496
Adjusted R² = 0.3926


In [36]:
# -------------------------------------------------------
# Compare 2-predictor vs 3-predictor LOG models
# on the same 33-state sample
# -------------------------------------------------------

log_two_predictors = [
    "vehicles_per_1000_population",
    "road_length_per_1000_population"
]

log_three_predictors = [
    "vehicles_per_1000_population",
    "road_length_per_1000_population",
    "population_density_2011"
]


# -------------------------------------------------------
# Prepare predictors
# -------------------------------------------------------

X_log_2 = log_df[
    log_two_predictors
].copy()

X_log_2 = sm.add_constant(X_log_2)

X_log_3 = log_df[
    log_three_predictors
].copy()

X_log_3 = sm.add_constant(X_log_3)


# -------------------------------------------------------
# Fit ordinary OLS models for nested comparison
# -------------------------------------------------------

log_acc_2 = sm.OLS(
    log_df["log_accident_rate"],
    X_log_2
).fit()

log_acc_3 = sm.OLS(
    log_df["log_accident_rate"],
    X_log_3
).fit()

log_fat_2 = sm.OLS(
    log_df["log_fatality_rate"],
    X_log_2
).fit()

log_fat_3 = sm.OLS(
    log_df["log_fatality_rate"],
    X_log_3
).fit()


# -------------------------------------------------------
# Model comparison table
# -------------------------------------------------------

log_comparison = pd.DataFrame({
    "Model": [
        "Log accident - 2 predictors",
        "Log accident - 3 predictors",
        "Log fatality - 2 predictors",
        "Log fatality - 3 predictors"
    ],
    "N": [
        len(log_df),
        len(log_df),
        len(log_df),
        len(log_df)
    ],
    "R2": [
        log_acc_2.rsquared,
        log_acc_3.rsquared,
        log_fat_2.rsquared,
        log_fat_3.rsquared
    ],
    "Adjusted_R2": [
        log_acc_2.rsquared_adj,
        log_acc_3.rsquared_adj,
        log_fat_2.rsquared_adj,
        log_fat_3.rsquared_adj
    ],
    "AIC": [
        log_acc_2.aic,
        log_acc_3.aic,
        log_fat_2.aic,
        log_fat_3.aic
    ],
    "BIC": [
        log_acc_2.bic,
        log_acc_3.bic,
        log_fat_2.bic,
        log_fat_3.bic
    ]
})

display(
    log_comparison.round(4)
)


# -------------------------------------------------------
# Nested F-tests
# -------------------------------------------------------

print("LOG ACCIDENT MODEL — ADDING POPULATION DENSITY")
print("=" * 60)

log_acc_f = log_acc_3.compare_f_test(
    log_acc_2
)

print(
    "F-statistic:",
    log_acc_f[0]
)

print(
    "p-value:",
    log_acc_f[1]
)

print(
    "Degrees of freedom difference:",
    log_acc_f[2]
)


print("\nLOG FATALITY MODEL — ADDING POPULATION DENSITY")
print("=" * 60)

log_fat_f = log_fat_3.compare_f_test(
    log_fat_2
)

print(
    "F-statistic:",
    log_fat_f[0]
)

print(
    "p-value:",
    log_fat_f[1]
)

print(
    "Degrees of freedom difference:",
    log_fat_f[2]
)

,Model,N,R2,Adjusted_R2,AIC,BIC
0,Log accident - 2 predictors,33,0.3486,0.3052,71.6247,76.1142
1,Log accident - 3 predictors,33,0.4631,0.4076,67.2458,73.2318
2,Log fatality - 2 predictors,33,0.2490,0.1990,47.7343,52.2239
3,Log fatality - 3 predictors,33,0.4496,0.3926,39.4838,45.4698


LOG ACCIDENT MODEL — ADDING POPULATION DENSITY
F-statistic: 6.18420181253746
p-value: 0.018891942095943758
Degrees of freedom difference: 1.0

LOG FATALITY MODEL — ADDING POPULATION DENSITY
F-statistic: 10.563974469901446
p-value: 0.002917858231001394
Degrees of freedom difference: 1.0


In [37]:
# -------------------------------------------------------
# Final Primary Regression Results
# 3-Predictor Models with HC3 Robust Inference
# -------------------------------------------------------

# -------------------------------------------------------
# Build accident-rate results
# -------------------------------------------------------

accident_results = pd.DataFrame({
    "Outcome": "Accident rate",
    "Variable": accident_model_3var.params.index,
    "Coefficient": accident_model_3var.params.values,
    "Robust_SE": accident_model_3var.bse.values,
    "t_value": accident_model_3var.tvalues.values,
    "p_value": accident_model_3var.pvalues.values
})

accident_results["R2"] = accident_model_3var.rsquared
accident_results["Adjusted_R2"] = accident_model_3var.rsquared_adj


# -------------------------------------------------------
# Build fatality-rate results
# -------------------------------------------------------

fatality_results = pd.DataFrame({
    "Outcome": "Fatality rate",
    "Variable": fatality_model_3var.params.index,
    "Coefficient": fatality_model_3var.params.values,
    "Robust_SE": fatality_model_3var.bse.values,
    "t_value": fatality_model_3var.tvalues.values,
    "p_value": fatality_model_3var.pvalues.values
})

fatality_results["R2"] = fatality_model_3var.rsquared
fatality_results["Adjusted_R2"] = fatality_model_3var.rsquared_adj


# -------------------------------------------------------
# Combine
# -------------------------------------------------------

final_regression_results = pd.concat(
    [
        accident_results,
        fatality_results
    ],
    ignore_index=True
)


# -------------------------------------------------------
# Round for presentation
# -------------------------------------------------------

final_regression_results_display = (
    final_regression_results
    .copy()
    .round({
        "Coefficient": 4,
        "Robust_SE": 4,
        "t_value": 4,
        "p_value": 4,
        "R2": 4,
        "Adjusted_R2": 4
    })
)

display(final_regression_results_display)

,Outcome,Variable,Coefficient,Robust_SE,t_value,p_value,R2,Adjusted_R2
0,Accident rate,const,-18.5255,19.3482,-0.9575,0.3383,0.5036,0.4539
1,Accident rate,vehicles_per_1000_population,2.9705,1.0025,2.9630,0.0030,0.5036,0.4539
2,Accident rate,road_length_per_1000_population,-0.5668,0.7690,-0.7371,0.4611,0.5036,0.4539
3,Accident rate,population_density_2011,-0.0058,0.0030,-1.9271,0.0540,0.5036,0.4539
4,Fatality rate,const,5.2924,3.0043,1.7616,0.0781,0.4844,0.4328
5,Fatality rate,vehicles_per_1000_population,0.3893,0.1343,2.8979,0.0038,0.4844,0.4328
6,Fatality rate,road_length_per_1000_population,-0.2242,0.2198,-1.0202,0.3076,0.4844,0.4328
7,Fatality rate,population_density_2011,-0.0012,0.0005,-2.3100,0.0209,0.4844,0.4328


In [38]:
# -------------------------------------------------------
# Save final primary regression results
# -------------------------------------------------------

final_regression_output = (
    "../data/research/"
    "final_3predictor_regression_results.csv"
)

final_regression_results.to_csv(
    final_regression_output,
    index=False
)

print(
    "Saved:",
    final_regression_output
)

Saved: ../data/research/final_3predictor_regression_results.csv


In [39]:
# -------------------------------------------------------
# Save log-model robustness results
# -------------------------------------------------------

log_robustness_results = pd.concat(
    [
        pd.DataFrame({
            "Outcome": "Log accident rate",
            "Variable": log_accident_model.params.index,
            "Coefficient": log_accident_model.params.values,
            "Robust_SE": log_accident_model.bse.values,
            "t_value": log_accident_model.tvalues.values,
            "p_value": log_accident_model.pvalues.values,
            "R2": log_accident_model.rsquared,
            "Adjusted_R2":
                log_accident_model.rsquared_adj
        }),

        pd.DataFrame({
            "Outcome": "Log fatality rate",
            "Variable": log_fatality_model.params.index,
            "Coefficient": log_fatality_model.params.values,
            "Robust_SE": log_fatality_model.bse.values,
            "t_value": log_fatality_model.tvalues.values,
            "p_value": log_fatality_model.pvalues.values,
            "R2": log_fatality_model.rsquared,
            "Adjusted_R2":
                log_fatality_model.rsquared_adj
        })
    ],
    ignore_index=True
)

log_output_path = (
    "../data/research/"
    "log_3predictor_regression_results.csv"
)

log_robustness_results.to_csv(
    log_output_path,
    index=False
)

print(
    "Saved:",
    log_output_path
)

Saved: ../data/research/log_3predictor_regression_results.csv
